In [1]:
#https://apframework.com/blog/essay/2025-12-16-LangChain-Deep-Agents-Skills

In [2]:
# pip install deepagents pyyaml

from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend  # 支持 execute，可运行 skills 脚本
from deepagents.backends.filesystem import FilesystemBackend

import os
from pathlib import Path
from langchain_openai import ChatOpenAI   # openai接口
from langchain_ollama import ChatOllama   # ollama接口

from langchain.agents import create_agent  # 创建智能体
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver  # 记忆
from langchain.messages import HumanMessage, AIMessage, SystemMessage

In [3]:
# openai接口
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen3.5-plus",             # 模型名称
    openai_api_key="sk-bvOWJIfQ8Y8IrZhzzpQZ80zSKHjWIHCkPXEgQ0H4Li5UCpUC",         # API Key
    openai_api_base="http://ai.wenmodel.com/v1", # API 地址 https://ai.wenmodel.com(https://m.tb.cn/h.ioaPOiq?tk=QGsd5gSbXZ)
    temperature=0.7,
    max_tokens=4096
)

response = llm.invoke("你是谁")
print(response.content)

你好！我是 **Qwen3.5**，是阿里巴巴集团最新推出的通义千问大语言模型。我能够回答问题、创作文字（比如写故事、写公文、写邮件、写剧本、逻辑推理、编程等），还能表达观点、玩游戏等。我支持全球 100 多种语言，具备强大的逻辑推理、数学计算及代码生成能力，可以处理复杂的多步骤任务。

如果你有任何问题或需要帮助，随时告诉我！😊 你想聊些什么呢？


# 2. 创建skills_agents


### create_agent与reate_deep_agent对比
| 功能     | `create_agent` 参数 | 调用方式                         | `create_deep_agent` 参数       | 调用方式                          | 区别                 |
| ------ | ----------------- | ---------------------------- | ---------------------------- | ----------------------------- | ------------------ |
| 模型     | `model`           | `model=llm`                  | `model`                      | `model=llm`                   | 相同                 |
| 工具     | `tools`           | `tools=[tool1,tool2]`        | `tools`                      | `tools=[tool1,tool2]`         | Deep Agent额外拥有内置工具 |
| 系统提示   | `system_prompt`   | `system_prompt="..."`        | `system_prompt`              | `system_prompt="..."`         | 相同                 |
| 中间件    | `middleware`      | `middleware=[xxx]`           | `middleware`                 | `middleware=[xxx]`            | 相同                 |
| 结构化输出  | `response_format` | `response_format=Schema`     | `response_format`            | `response_format=Schema`      | 相同                 |
| 状态管理   | `state_schema`    | `state_schema=MyState`       | 内部管理                         | 自动                            | Deep Agent封装       |
| 上下文    | `context_schema`  | `context_schema=Context`     | 内部管理                         | 自动                            | Deep Agent封装       |
| 记忆    | `checkpointer`    | `checkpointer=InMemorySaver()` | 支持                           | `checkpointer=...`            | 长任务恢复              |
| 存储     | `store`           | `store=store`                | 支持                           | `store=store`                 | 长期记忆               |
| 调试     | `debug`           | `debug=True`                 | 支持                           | `debug=True`                  | 相同                 |
| 名称     | `name`            | `name="agent"`               | 支持                           | `name="agent"`                | 相同                 |
| 子Agent | ❌                 | 无                            | `subagents`                  | `subagents=[...]`             | Deep Agent独有       |
| 技能     | ❌                 | 无                            | `skills`                     | `skills=["./skills"]`         | Claude Skill支持     |
| 文件系统   | ❌                 | 无                            | `backend`                    | `backend=FilesystemBackend()` | Deep Agent核心       |
| 权限控制   | ❌                 | 无                            | `interrupt_on` / permissions | 配置权限                          | 安全执行               |



# 1.Skill Loader
![image.png](attachment:82c79cbb-bfa9-4351-9575-efa6b901d780.png)

In [4]:
from pathlib import Path
from deepagents.backends import LocalShellBackend

# 1. 路径：notebook 中用 cwd；skills 对 backend 使用 POSIX 虚拟路径
# current_dir = Path(__file__).parent.resolve()  # .py 中
current_dir = Path.cwd()  # .ipynb 中
SKILLS_HOST_DIR = current_dir / "skills"
SKILLS_VIRTUAL = "/skills/"  # create_deep_agent(skills=...) 相对 backend root

assert SKILLS_HOST_DIR.is_dir(), f"skills 目录不存在: {SKILLS_HOST_DIR}"

# 2. LocalShellBackend = 文件系统工具 + execute（跑 skills/scripts）
# inherit_env=True 才能找到本机 python / node / npm
backend = LocalShellBackend(
    root_dir=str(current_dir),
    virtual_mode=True,
    inherit_env=True,
    timeout=300,
)

print("root:", current_dir)
print("skills host:", SKILLS_HOST_DIR)
print("skills virtual:", SKILLS_VIRTUAL)
print("skills:", [p.name for p in SKILLS_HOST_DIR.iterdir() if p.is_dir()])

root: e:\code\jupyter\智能体学习\langchain
skills host: e:\code\jupyter\智能体学习\langchain\skills
skills virtual: /skills/
skills: ['docx', 'pdf', 'pptx', 'skill-creator', 'xlsx']


## 2.1 构建 Skills 脚本工具

辅助工具：列出 skill 脚本、按路径执行脚本。  
配合 `LocalShellBackend` 自带的 `execute` / `read_file`，`skills_agent` 即可按 `SKILL.md` 完成任务。


In [5]:
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import yaml


def _resolve_bin(name: str) -> str:
    """Windows 上 npm/npx 是 .cmd，subprocess 无 shell 时必须显式指定。"""
    if sys.platform == "win32":
        found = shutil.which(f"{name}.cmd") or shutil.which(name)
        return found or f"{name}.cmd"
    return shutil.which(name) or name


from langchain.tools import tool

SCRIPT_SUFFIXES = {".py", ".js", ".mjs", ".sh", ".bat", ".ps1", ".ts"}
OUTPUT_SUFFIXES = {".docx", ".pdf", ".pptx", ".xlsx", ".csv", ".png", ".md", ".txt", ".json", ".zip"}


@dataclass
class SkillInfo:
    name: str
    description: str
    path: Path
    skill_md: Path
    content: str


class ClaudeSkillRuntime:
    """Claude Skills 执行运行时：说明书(SKILL.md) + 代码执行 + 产物检测。"""

    def __init__(self, skills_dir: str | Path, work_dir: str | Path | None = None):
        self.skills_dir = Path(skills_dir).resolve()
        self.work_dir = Path(work_dir or Path.cwd()).resolve()
        self.skills: dict[str, SkillInfo] = {}
        self._started_at = time.time()
        self.reload()

    def reload(self) -> None:
        self.skills.clear()
        if not self.skills_dir.is_dir():
            return
        for path in sorted(self.skills_dir.iterdir()):
            md = path / "SKILL.md"
            if not path.is_dir() or not md.exists():
                continue
            content = md.read_text(encoding="utf-8")
            meta: dict[str, Any] = {}
            m = re.match(r"^---\s*\n(.*?)\n---\s*\n", content, re.DOTALL)
            if m:
                try:
                    meta = yaml.safe_load(m.group(1)) or {}
                except Exception:
                    meta = {}
            name = str(meta.get("name") or path.name)
            self.skills[name] = SkillInfo(
                name=name,
                description=str(meta.get("description") or "").strip(),
                path=path,
                skill_md=md,
                content=content,
            )

    def list_skills(self) -> list[dict[str, str]]:
        return [
            {
                "name": s.name,
                "description": s.description,
                "skill_md": f"/skills/{s.path.name}/SKILL.md",
                "host_path": str(s.path),
            }
            for s in self.skills.values()
        ]

    def load_skill(self, name: str) -> dict[str, Any]:
        skill = self._get(name)
        return {
            "name": skill.name,
            "description": skill.description,
            "skill_md_virtual": f"/skills/{skill.path.name}/SKILL.md",
            "host_path": str(skill.path),
            "content": skill.content,
            "scripts": self.list_scripts(name),
            "hint": (
                "这是说明书，不是自动执行器。"
                "请按 content 编写代码，再用 run_js_code / run_python_code / create_docx_file 真正执行并生成文件。"
            ),
        }

    def list_scripts(self, name: str) -> list[str]:
        skill = self._get(name)
        scripts = []
        for f in sorted(skill.path.rglob("*")):
            if not f.is_file() or f.suffix.lower() not in SCRIPT_SUFFIXES:
                continue
            if any(p in {"node_modules", ".git", "__pycache__"} for p in f.parts):
                continue
            scripts.append(f.relative_to(skill.path).as_posix())
        return scripts

    def run_script(self, name: str, script_relpath: str, args: str = "") -> dict[str, Any]:
        skill = self._get(name)
        script = (skill.path / script_relpath).resolve()
        if not str(script).startswith(str(skill.path.resolve())):
            return {"ok": False, "error": "脚本路径越界"}
        if not script.is_file():
            return {"ok": False, "error": f"脚本不存在: {script_relpath}"}
        arg_list = shlex.split(args, posix=os.name != "nt") if args.strip() else []
        cmd = self._cmd_for_script(script, arg_list)
        return self._run(cmd, cwd=self.work_dir, extra={"skill": name, "script": script_relpath})

    def run_python_code(self, code: str, filename: str = "_skill_runtime_tmp.py") -> dict[str, Any]:
        path = self._write_code(filename, code)
        before = self._output_snapshot()
        result = self._run(["python", str(path)], cwd=self.work_dir, extra={"file": str(path)})
        result["new_files"] = self._new_outputs(before)
        return result

    def run_js_code(self, code: str, filename: str = "_skill_runtime_tmp.js") -> dict[str, Any]:
        """按 docx 等 skill 指南执行 JS（工作目录需已 npm install docx）。"""
        self.ensure_js_deps(["docx"])
        path = self._write_code(filename, code)
        before = self._output_snapshot()
        result = self._run(["node", str(path)], cwd=self.work_dir, extra={"file": str(path)})
        result["new_files"] = self._new_outputs(before)
        return result

    def ensure_js_deps(self, packages: list[str]) -> dict[str, Any]:
        pkg_json = self.work_dir / "package.json"
        node_modules = self.work_dir / "node_modules"
        logs = []
        npm = _resolve_bin("npm")
        if not pkg_json.exists():
            r = subprocess.run([npm, "init", "-y"], cwd=str(self.work_dir), capture_output=True, text=True)
            logs.append({"step": "npm init", "returncode": r.returncode, "stderr": r.stderr[-500:]})
        missing = [p for p in packages if not (node_modules / p).exists()]
        if missing:
            r = subprocess.run(
                [npm, "install", *missing, "--save"],
                cwd=str(self.work_dir),
                capture_output=True,
                text=True,
                timeout=300,
            )
            logs.append(
                {
                    "step": f"npm install {' '.join(missing)}",
                    "returncode": r.returncode,
                    "stderr": r.stderr[-800:],
                }
            )
        return {"ok": True, "missing_installed": missing, "logs": logs}

    def list_outputs(self, since_epoch: float | None = None) -> list[str]:
        since = since_epoch if since_epoch is not None else self._started_at
        files = []
        for f in self.work_dir.rglob("*"):
            if not f.is_file():
                continue
            if "skills" in f.parts or "node_modules" in f.parts or ".git" in f.parts:
                continue
            if f.suffix.lower() not in OUTPUT_SUFFIXES:
                continue
            if f.stat().st_mtime >= since - 1:
                files.append(str(f.relative_to(self.work_dir)).replace("\\", "/"))
        return sorted(files)

    def create_docx(
        self,
        output_filename: str,
        title: str,
        paragraphs: list[str] | None = None,
        bullet_points: list[str] | None = None,
    ) -> dict[str, Any]:
        """docx skill 的可靠 Runtime 落盘入口（python-docx），保证一定生成文件。"""
        try:
            from docx import Document
            from docx.enum.text import WD_ALIGN_PARAGRAPH
        except ImportError:
            return {"ok": False, "error": "缺少 python-docx，请先 pip install python-docx"}

        paragraphs = paragraphs or []
        bullet_points = bullet_points or []
        out = (self.work_dir / output_filename).resolve()
        if out.suffix.lower() != ".docx":
            out = out.with_suffix(".docx")
        if not str(out).startswith(str(self.work_dir)):
            return {"ok": False, "error": "输出路径必须在工作目录内"}

        doc = Document()
        h = doc.add_heading(title, level=1)
        h.alignment = WD_ALIGN_PARAGRAPH.CENTER
        for p in paragraphs:
            doc.add_paragraph(str(p))
        for b in bullet_points:
            doc.add_paragraph(str(b), style="List Bullet")
        doc.save(str(out))
        rel = str(out.relative_to(self.work_dir)).replace("\\", "/")
        return {"ok": True, "file": rel, "abs_path": str(out), "size": out.stat().st_size}

    def _get(self, name: str) -> SkillInfo:
        if name not in self.skills:
            raise KeyError(f"Skill 不存在: {name}；可用: {list(self.skills)}")
        return self.skills[name]

    def _write_code(self, filename: str, code: str) -> Path:
        path = (self.work_dir / Path(filename).name).resolve()
        path.write_text(code, encoding="utf-8")
        return path

    def _cmd_for_script(self, script: Path, arg_list: list[str]) -> list[str]:
        suffix = script.suffix.lower()
        if suffix == ".py":
            return ["python", str(script), *arg_list]
        if suffix in {".js", ".mjs"}:
            return ["node", str(script), *arg_list]
        if suffix == ".ts":
            return [_resolve_bin("npx"), "--yes", "tsx", str(script), *arg_list]
        if suffix == ".ps1":
            return ["powershell", "-ExecutionPolicy", "Bypass", "-File", str(script), *arg_list]
        if suffix == ".bat":
            return [str(script), *arg_list]
        if suffix == ".sh":
            return ["bash", str(script), *arg_list]
        raise ValueError(f"不支持的脚本类型: {suffix}")

    def _run(self, cmd: list[str], cwd: Path, extra: dict | None = None) -> dict[str, Any]:
        try:
            r = subprocess.run(
                cmd,
                cwd=str(cwd),
                capture_output=True,
                text=True,
                timeout=300,
                env={**os.environ, "WORK_DIR": str(self.work_dir), "SKILLS_DIR": str(self.skills_dir)},
            )
            out = {
                "ok": r.returncode == 0,
                "cmd": cmd,
                "returncode": r.returncode,
                "stdout": (r.stdout or "")[-8000:],
                "stderr": (r.stderr or "")[-8000:],
                "cwd": str(cwd),
            }
            if extra:
                out.update(extra)
            return out
        except subprocess.TimeoutExpired:
            return {"ok": False, "error": "执行超时(>300s)", "cmd": cmd}
        except Exception as e:
            return {"ok": False, "error": str(e), "cmd": cmd}

    def _output_snapshot(self) -> set[str]:
        return set(self.list_outputs(since_epoch=0))

    def _new_outputs(self, before: set[str]) -> list[str]:
        after = set(self.list_outputs(since_epoch=0))
        return sorted(after - before)


# 全局 Runtime（skills_agent 工具共享）
runtime = ClaudeSkillRuntime(SKILLS_HOST_DIR, current_dir)
print("skills:", [s["name"] for s in runtime.list_skills()])
print("work_dir:", runtime.work_dir)


# ---------- LangChain Tools（挂到 skills_agent） ----------

@tool
def list_available_skills() -> str:
    """列出所有可用 Claude Skills。"""
    return json.dumps(runtime.list_skills(), ensure_ascii=False, indent=2)


@tool
def load_skill(skill_name: str) -> str:
    """加载指定 Skill 的完整 SKILL.md 说明书与脚本列表。这是执行前的必做步骤。"""
    try:
        return json.dumps(runtime.load_skill(skill_name), ensure_ascii=False, indent=2)
    except Exception as e:
        return json.dumps({"ok": False, "error": str(e)}, ensure_ascii=False)


@tool
def list_skill_scripts(skill_name: str) -> str:
    """列出 skill 目录下可执行脚本（相对路径）。"""
    try:
        return json.dumps(
            {"skill": skill_name, "scripts": runtime.list_scripts(skill_name)},
            ensure_ascii=False,
            indent=2,
        )
    except Exception as e:
        return json.dumps({"ok": False, "error": str(e)}, ensure_ascii=False)


@tool
def run_skill_script(skill_name: str, script_relpath: str, script_args: str = "") -> str:
    """【Runtime 主入口】选中 skill 后必须调用本工具执行该 skill 目录内脚本并落盘。

    例：run_skill_script('docx', 'create_deep_agent_params.js')
    script_relpath 相对 skill 根目录；script_args 为可选命令行参数。
    """
    return json.dumps(runtime.run_script(skill_name, script_relpath, script_args), ensure_ascii=False, indent=2)


@tool
def run_js_code(code: str, filename: str = "_skill_runtime_tmp.js") -> str:
    """将 JavaScript 写入工作目录并执行（docx 技能创建文档可用）。会自动 npm install docx。必须写出生成文件的代码。"""
    return json.dumps(runtime.run_js_code(code, filename), ensure_ascii=False, indent=2)


@tool
def run_python_code(code: str, filename: str = "_skill_runtime_tmp.py") -> str:
    """将 Python 写入工作目录并执行。适合调用 skill 脚本或自定义逻辑。"""
    return json.dumps(runtime.run_python_code(code, filename), ensure_ascii=False, indent=2)


@tool
def create_docx_file(
    output_filename: str,
    title: str,
    paragraphs_json: str = "[]",
    bullet_points_json: str = "[]",
) -> str:
    """docx skill 的可靠落盘工具：用 python-docx 直接生成 .docx。创建 Word 文档时优先用这个，保证文件一定生成。

    Args:
        output_filename: 如 deep_agent_params.docx
        title: 文档标题
        paragraphs_json: JSON 字符串数组，正文段落
        bullet_points_json: JSON 字符串数组，项目符号列表
    """
    try:
        paragraphs = json.loads(paragraphs_json) if paragraphs_json else []
        bullets = json.loads(bullet_points_json) if bullet_points_json else []
        if not isinstance(paragraphs, list):
            paragraphs = [str(paragraphs)]
        if not isinstance(bullets, list):
            bullets = [str(bullets)]
    except json.JSONDecodeError as e:
        return json.dumps({"ok": False, "error": f"JSON 解析失败: {e}"}, ensure_ascii=False)
    return json.dumps(
        runtime.create_docx(output_filename, title, paragraphs, bullets),
        ensure_ascii=False,
        indent=2,
    )


@tool
def list_generated_files() -> str:
    """列出 Runtime 启动后工作目录新生成的产物文件。任务结束前必须调用以确认文件已生成。"""
    return json.dumps(
        {"files": runtime.list_outputs(), "work_dir": str(runtime.work_dir)},
        ensure_ascii=False,
        indent=2,
    )


skill_tools = [
    list_available_skills,
    load_skill,
    list_skill_scripts,
    run_skill_script,
    run_js_code,
    run_python_code,
    create_docx_file,
    list_generated_files,
]
print("skill tools:", [t.name for t in skill_tools])


skills: ['docx', 'pdf', 'pptx', 'skill-creator', 'xlsx']
work_dir: E:\code\jupyter\智能体学习\langchain
skill tools: ['list_available_skills', 'load_skill', 'list_skill_scripts', 'run_skill_script', 'run_js_code', 'run_python_code', 'create_docx_file', 'list_generated_files']


## 2.2 创建 Skills Agent（挂 Runtime）

核心链路（你的解决想法）：

```
用户请求
   ↓
skills_agent 判断要用哪个 skill（list / load）
   ↓
Runtime @tool 执行 skill 内脚本（真正落盘）
   ↓
list_generated_files 验收
```

- **判断层**：`list_available_skills` / `load_skill`
- **执行层（Runtime）**：`run_skill_script` / `run_js_code` / `run_python_code` / `create_docx_file`
- **验收层**：`list_generated_files`

注意：`create_deep_agent` 自带的 `read_file` 等会抢执行权，下面会用 `_ToolExclusionMiddleware` 屏蔽它们，强制走 Runtime。


In [6]:
# Skills Agent 系统提示
# 链路：选 skill → Runtime @tool 执行脚本 → 验收产物

SKILLS_AGENT_PROMPT = """你是 skills_agent：负责「选 skill + 调 Runtime 执行」。

## 固定两步（缺一不可）
1. **判断**：list_available_skills → load_skill(skill_name)
2. **执行（必须 tool call，禁止只口头说「我将执行」）**：
   - skill 目录已有脚本 → 立刻 run_skill_script(skill_name, script_relpath)
   - 无现成脚本、要生成 Word → create_docx_file 或 run_js_code
   - 其他 → run_python_code / run_js_code
3. **验收**：list_generated_files；无产物则根据 stderr 修复再执行

## 禁止
- 用 read_file / execute 通读或旁路执行（这些内置工具已被屏蔽）
- Windows 绝对路径；只用相对工作目录或 skill 相对脚本路径
- 只描述不调用 Runtime 工具

## 约束
- 产物写在工作目录根，不要写进 skills/
- 最终回复给出文件相对路径与大小
"""


## 3.2 创建记忆checkpointer = MemorySaver()

In [7]:
from langgraph.checkpoint.memory import MemorySaver 

# 记忆
checkpointer = MemorySaver()

# 创建config
config = {"configurable": {"thread_id": "demo-001"}}

## 3.3 创建 skills_agent

直接创建 `skills_agent`，挂载 Runtime 工具与 skills，不再使用子代理委派。


In [8]:
from deepagents import create_deep_agent
from deepagents.middleware._tool_exclusion import _ToolExclusionMiddleware

# 屏蔽 deep agent 内置文件/壳工具，避免抢 Runtime 的执行权
# （上次失败就是模型反复 read_file 读脚本，却从不 run_skill_script）
EXCLUDE_COMPETING_TOOLS = frozenset({
    "read_file",
    "write_file",
    "edit_file",
    "execute",
})

skills_agent = create_deep_agent(
    model=llm,
    backend=backend,
    tools=skill_tools,  # Runtime @tools：选 skill + 执行脚本
    skills=[SKILLS_VIRTUAL],
    checkpointer=checkpointer,
    system_prompt=SKILLS_AGENT_PROMPT,
    middleware=[_ToolExclusionMiddleware(excluded=EXCLUDE_COMPETING_TOOLS)],
    name="skills-agent",
)

print("skills_agent ready")
print("runtime tools:", [t.name for t in skill_tools])
print("excluded competing tools:", sorted(EXCLUDE_COMPETING_TOOLS))


skills_agent ready
runtime tools: ['list_available_skills', 'load_skill', 'list_skill_scripts', 'run_skill_script', 'run_js_code', 'run_python_code', 'create_docx_file', 'list_generated_files']
excluded competing tools: ['edit_file', 'execute', 'read_file', 'write_file']


## 3.4 构造消息调用智能体

In [9]:
# 多模态msg构造函数

import base64
from langchain.messages import HumanMessage


def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()


def build_msg(text=None, image_path=None):

    content = []

    # 添加文本
    if text:
        content.append({
            "type": "text",
            "text": text
        })

    # 添加图片
    if image_path:
        image_data = encode_image(image_path)

        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{image_data}",
                "detail": "auto"
            }
        })

    return [
        HumanMessage(content=content)
    ]



# 流式输入
def chat_stream(agent, message, config=None):
    result = agent.stream(
        {"messages":message}, 
        stream_mode='updates',
        config=config
    )
    for chunk in result:
        for node_name, update in chunk.items():
            
            # 新增空判断
            if not update:
                continue
            
            if "messages" in update:
                for msg in update["messages"]:
                    
                    # system提示词
                    if msg.type == "system":
                       print(f"[{msg.type}] {msg.content}")  
                   
                    # user文本内容
                    if msg.type == "human":
                       print(f"[{msg.type}] {msg.content}")  
                    
                    # ai回复
                    if msg.type == "ai":
                        if msg.content:
                            print(f"[{msg.type}] {msg.content}")
                        
                        if hasattr(msg, 'tool_calls') and msg.tool_calls:
                            calls = [tc['name'] for tc in msg.tool_calls]
                            print(f"请求调用: {calls}")
    
                    # tool工具回复
                    if msg.type == "tool":
                        print(f"[调用工具 {msg.name}]")
                        print(f"[{msg.type}] {msg.content}") 


In [10]:
# 查询可用 skills
msg = build_msg(text="你有哪些skills？请查询后回答", image_path=None)
chat_stream(skills_agent, msg, config)


请求调用: ['list_available_skills']
[调用工具 list_available_skills]
[tool] [
  {
    "name": "docx",
    "description": "Use this skill whenever the user wants to create, read, edit, or manipulate Word documents (.docx files). Triggers include: any mention of \"Word doc\", \"word document\", \".docx\", or requests to produce professional documents with formatting like tables of contents, headings, page numbers, or letterheads. Also use when extracting or reorganizing content from .docx files, inserting or replacing images in documents, performing find-and-replace in Word files, working with tracked changes or comments, or converting content into a polished Word document. If the user asks for a \"report\", \"memo\", \"letter\", \"template\", or similar deliverable as a Word or .docx file, use this skill. Do NOT use for PDFs, spreadsheets, Google Docs, or general coding tasks unrelated to document generation.",
    "skill_md": "/skills/docx/SKILL.md",
    "host_path": "E:\\code\\jupyter\\智能体学习\

In [12]:
# 使用 docx skill 生成文档
msg = build_msg(
    text=(
        "请使用 ppt 技能，在当前目录生成一份 ppt 文档 "
        "deep_agent_params.ppt：一页内容，介绍 create_deep_agent 的主要参数"
    ),
    image_path=None,
)
chat_stream(skills_agent, msg, config)


请求调用: ['load_skill']
[调用工具 load_skill]
[tool] {
  "name": "pptx",
  "description": "Use this skill any time a .pptx file is involved in any way — as input, output, or both. This includes: creating slide decks, pitch decks, or presentations; reading, parsing, or extracting text from any .pptx file (even if the extracted content will be used elsewhere, like in an email or summary); editing, modifying, or updating existing presentations; combining or splitting slide files; working with templates, layouts, speaker notes, or comments. Trigger whenever the user mentions \"deck,\" \"slides,\" \"presentation,\" or references a .pptx filename, regardless of what they plan to do with the content afterward. If a .pptx file needs to be opened, created, or touched, use this skill.",
  "skill_md_virtual": "/skills/pptx/SKILL.md",
  "host_path": "E:\\code\\jupyter\\智能体学习\\langchain\\skills\\pptx",
  "content": "---\nname: pptx\ndescription: \"Use this skill any time a .pptx file is involved in any wa

---

## 架构小结

```
用户请求
   │
   ▼
skills_agent（判断用哪个 skill）
   │  list_available_skills / load_skill
   ▼
ClaudeSkillRuntime（@tool 真正执行）
   ├─ run_skill_script   ← 执行 skills/<name>/ 下脚本（主路径）
   ├─ run_js_code / run_python_code
   ├─ create_docx_file
   └─ list_generated_files（验收落盘）
```

关键点：
1. Skills 的 `SKILL.md` 只是说明书，**不会自动跑**
2. 必须用 `@tool` 包一层 Runtime，才会执行脚本并生成文件
3. 要排除 deep agent 自带 `read_file` 等干扰，否则容易「只读不执行」
